# JobKB — inspection & QA
Quick spot-checks over the `kb/` outputs of `run_pipeline.py`.

In [ ]:
import os, pandas as pd
pd.set_option('display.max_colwidth', 80)
KB = os.path.join('..', 'kb')
def load(name): return pd.read_csv(os.path.join(KB, name), dtype=str, keep_default_na=False)
occ   = load('occupations.csv')
skl   = load('skills.csv')
align = load('concept_alignments.csv')
uocc  = load('unified_occupations.csv')
uskl  = load('unified_skills.csv')
prov  = load('provenance.csv')
prov

In [ ]:
# Counts per source
print('Occupations by source:'); print(occ['source'].value_counts())
print('\nSkills by source:');     print(skl['source'].value_counts())
print('\nEN label coverage (real occ):',
      (occ[occ.occupation_type!='isco_group'].pref_label_en!='').sum(),
      '/', (occ.occupation_type!='isco_group').sum())

In [ ]:
# De-duplication: unified occupations that merged >1 source entity
uocc['n_members'] = uocc.member_entity_ids.str.split(' | ').map(len)
merged = uocc[uocc.n_members > 1].sort_values('n_members', ascending=False)
print(f'{len(merged)} multi-source unified occupations of {len(uocc)} total')
merged[['primary_label_en','primary_label_fr','isco_code','sources','n_members']].head(25)

In [ ]:
# De-duplication: unified skills that merged >1 source entity (e.g. Python, SQL)
uskl['n_members'] = uskl.member_entity_ids.str.split(' | ').map(len)
ms = uskl[uskl.n_members > 1].sort_values('n_members', ascending=False)
print(f'{len(ms)} multi-source unified skills of {len(uskl)} total')
ms[['primary_label_en','primary_label_fr','hard_soft','it_subtype','sources','n_members']].head(25)

In [ ]:
# Sample exactMatch alignments across sources
ex = align[align.relation=='skos:exactMatch']
print('relations:'); print(align.relation.value_counts())
ex[['source_a','source_b','confidence','method','notes']].sample(min(20, len(ex)), random_state=0)

In [ ]:
# Hard/soft balance and IT subtype distribution
print('hard/soft:'); print(skl.hard_soft_provisional.value_counts())
print('\nIT subtype:'); print(skl.it_subtype.value_counts())